# BTC Price Forecasting

## Model Training and Evaluation

This notebook trains and evaluates a deep learning model for Bitcoin price forecasting using preprocessed time-series data.

The workflow includes dataset loading, sequence construction, model training, test evaluation, visual analysis, and export of the trained model and prediction outputs.

### Required input files

The notebook expects the following files to be available in the Kaggle environment:

- `train_scaled.npy`
- `val_scaled.npy`
- `test_scaled.npy`
- `scaler.pkl`
- `processed_btc_features.csv` *(optional, used for timestamps)*

### Forecasting objective

The model uses the previous **1,440 minutes** of data as input and predicts the BTC closing price **60 minutes ahead**.


## 1. Environment and Data Setup

This section imports the required libraries, defines the working directories, sets the forecasting configuration, and loads the preprocessed training, validation, and test arrays.

The setup is designed to automatically locate the processed dataset inside the Kaggle input or working directories.


In [57]:
from pathlib import Path
import os
import json
import pickle
import joblib

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

KAGGLE_INPUT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
OUT_DIR = WORK_DIR / "btc_model_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_SIZE   = 1440
HORIZON       = 60
BATCH_SIZE    = 32
LEARNING_RATE = 0.001
CLOSE_IDX     = 0

REQUIRED_FILES = [
    "train_scaled.npy",
    "val_scaled.npy",
    "test_scaled.npy",
    "scaler.pkl"
]

def find_processed_dir():
    candidate_dirs = [
        WORK_DIR,
        KAGGLE_INPUT / "processed_btc" / "processed_btc",
        KAGGLE_INPUT / "processed_btc",
    ]

    for d in candidate_dirs:
        if all((d / f).exists() for f in REQUIRED_FILES):
            return d

    for d in KAGGLE_INPUT.rglob("*"):
        if d.is_dir() and all((d / f).exists() for f in REQUIRED_FILES):
            return d

    raise FileNotFoundError(
        "Could not find required processed files. Expected files: "
        + ", ".join(REQUIRED_FILES)
        + "\nChecked /kaggle/working and /kaggle/input."
    )

PROCESSED_DIR = find_processed_dir()

print("Using PROCESSED_DIR:", PROCESSED_DIR)
print("Saving outputs to  :", OUT_DIR)

print("\nLoading scaled arrays...")

train_scaled = np.load(PROCESSED_DIR / "train_scaled.npy")
val_scaled   = np.load(PROCESSED_DIR / "val_scaled.npy")
test_scaled  = np.load(PROCESSED_DIR / "test_scaled.npy")

print(f"  train_scaled : {train_scaled.shape}")
print(f"  val_scaled   : {val_scaled.shape}")
print(f"  test_scaled  : {test_scaled.shape}")

scaler_path = PROCESSED_DIR / "scaler.pkl"

try:
    scaler = joblib.load(scaler_path)
except Exception:
    with open(scaler_path, "rb") as f:
        scaler = pickle.load(f)

print(f"\nScaler Close range: [{scaler.data_min_[CLOSE_IDX]:.2f}, {scaler.data_max_[CLOSE_IDX]:.2f}] USD")
print(f"TensorFlow version : {tf.__version__}")
print(f"GPU available      : {tf.config.list_physical_devices('GPU')}")

features_path = PROCESSED_DIR / "processed_btc_features.csv"
if features_path.exists():
    df_features = pd.read_csv(features_path)
    print(f"\nLoaded features file: {features_path}")
    print("Features shape:", df_features.shape)
else:
    df_features = None
    print("\nNo processed_btc_features.csv found. Continuing without it.")

print("\nSetup complete ✓")


Using PROCESSED_DIR: /kaggle/working
Saving outputs to  : /kaggle/working/btc_model_outputs

Loading scaled arrays...
  train_scaled : (786240, 6)
  val_scaled   : (132480, 6)
  test_scaled  : (142447, 6)

Scaler Close range: [0.06, 19891.99] USD
TensorFlow version : 2.19.0
GPU available      : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]

Loaded features file: /kaggle/working/processed_btc_features.csv
Features shape: (1061167, 7)

Setup complete ✓


## 2. Time-Series Dataset Construction

This section creates TensorFlow data pipelines from the scaled arrays.

Each sample contains a historical input window and a future target value. The pipeline streams the windows efficiently, which avoids loading all possible sequences into memory at once.


In [58]:
def make_dataset(data, window_size, horizon, batch_size, shuffle=False, shift=60):
    """Stream sliding windows directly from a scaled array without RAM blow-up."""
    data = np.asarray(data, dtype=np.float32)

    total_length = window_size + horizon
    if len(data) < total_length:
        raise ValueError(
            f"Data has only {len(data)} rows, but needs at least {total_length} rows "
            f"for WINDOW_SIZE={window_size} and HORIZON={horizon}."
        )

    dataset = tf.data.Dataset.from_tensor_slices(data)
    dataset = dataset.window(total_length, shift=shift, drop_remainder=True)
    dataset = dataset.flat_map(lambda w: w.batch(total_length, drop_remainder=True))

    dataset = dataset.map(
        lambda w: (w[:window_size], w[window_size + horizon - 1, CLOSE_IDX]),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if shuffle:
        dataset = dataset.shuffle(buffer_size=2000, seed=42, reshuffle_each_iteration=True)

    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

print("Building tf.data pipelines (shift=60)...")

train_ds = make_dataset(train_scaled, WINDOW_SIZE, HORIZON, BATCH_SIZE, shuffle=True,  shift=60)
val_ds   = make_dataset(val_scaled,   WINDOW_SIZE, HORIZON, BATCH_SIZE, shuffle=False, shift=60)
test_ds  = make_dataset(test_scaled,  WINDOW_SIZE, HORIZON, BATCH_SIZE, shuffle=False, shift=60)

for name, ds in [("train_ds", train_ds), ("val_ds", val_ds), ("test_ds", test_ds)]:
    for X_batch, y_batch in ds.take(1):
        print(f"  {name} — X: {X_batch.shape}  y: {y_batch.shape}")

print("Pipelines ready ✓")


Building tf.data pipelines (shift=60)...
  train_ds — X: (32, 1440, 6)  y: (32,)
  val_ds — X: (32, 1440, 6)  y: (32,)
  test_ds — X: (32, 1440, 6)  y: (32,)
Pipelines ready ✓


## 3. Pipeline Inspection

This section checks the generated batches before training.

The inspection confirms the input shape, target shape, and value ranges for the training, validation, and test datasets.


In [59]:
print('Inspecting pipelines...')
print()
for name, ds in [('train_ds', train_ds), ('val_ds', val_ds), ('test_ds', test_ds)]:
    for X_batch, y_batch in ds.take(1):
        print(f'  {name}')
        print(f'    X shape  : {X_batch.shape}  → (batch, timesteps, features)')
        print(f'    y shape  : {y_batch.shape}  → (batch,)')
        print(f'    X range  : [{X_batch.numpy().min():.4f}, {X_batch.numpy().max():.4f}]')
        print(f'    y range  : [{y_batch.numpy().min():.4f}, {y_batch.numpy().max():.4f}]')
        print()

print('Expected X: (32, 1440, 6)   Expected y: (32,)')
print('Inspection complete ✓')


Inspecting pipelines...

  train_ds
    X shape  : (32, 1440, 6)  → (batch, timesteps, features)
    y shape  : (32,)  → (batch,)
    X range  : [0.0000, 1.0000]
    y range  : [0.0389, 0.0647]

  val_ds
    X shape  : (32, 1440, 6)  → (batch, timesteps, features)
    y shape  : (32,)  → (batch,)
    X range  : [0.0000, 1.0000]
    y range  : [0.3163, 0.3342]

  test_ds
    X shape  : (32, 1440, 6)  → (batch, timesteps, features)
    y shape  : (32,)  → (batch,)
    X range  : [0.0000, 1.0000]
    y range  : [0.3225, 0.3308]

Expected X: (32, 1440, 6)   Expected y: (32,)
Inspection complete ✓


## 4. Model Architecture

This section defines a stacked LSTM model for sequence forecasting.

The model receives a 24-hour time window and learns temporal dependencies across the BTC feature sequence. Dropout layers are included to reduce overfitting during training.

| Component | Purpose |
|---|---|
| Input layer | Receives the historical time window |
| LSTM layers | Learn short-term and long-term temporal patterns |
| Dropout layers | Improve generalisation |
| Dense output layer | Produces the future scaled closing-price prediction |


In [60]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense

n_features = train_scaled.shape[1]

model = Sequential([
    Input(shape=(WINDOW_SIZE, n_features)),
    LSTM(128, return_sequences=True),
    Dropout(0.2),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.1),
    Dense(1)
])

model.compile(
    loss='mse',
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    metrics=['mae']
)

model.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_6 (LSTM)                   │ (None, 1440, 128)      │        69,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 1440, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ (None, 1440, 64)       │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 1440, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_8 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 130,977 (511.63 KB)

 Trainable params: 130,977 (511.63 KB)

 Non-trainable params: 0 (0.00 B)

## 5. Model Training

This section trains the LSTM model on the training dataset and validates it on the validation dataset.

The training process uses early stopping, best-model checkpointing, and learning-rate reduction to improve stability and preserve the best validation result.


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

model_path = OUT_DIR / "model.keras"

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=str(model_path),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

print("Starting training...")
print(f"  Model will be saved → {model_path}")
print()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks,
    verbose=1
)

print()
print("Training complete ✓")

best_epoch = int(np.argmin(history.history["val_loss"]))
best_val_loss = history.history["val_loss"][best_epoch]

print(f"  Best epoch     : {best_epoch + 1}")
print(f"  Best val loss  : {best_val_loss:.6f}")


Starting training...
  Model will be saved → /kaggle/working/btc_model_outputs/model.keras

Epoch 1/50
    409/Unknown 56s 122ms/step - loss: 0.0021 - mae: 0.0309
Epoch 1: val_loss improved from inf to 0.00020, saving model to /kaggle/working/btc_model_outputs/model.keras
409/409 ━━━━━━━━━━━━━━━━━━━━ 60s 133ms/step - loss: 0.0021 - mae: 0.0309 - val_loss: 2.0171e-04 - val_mae: 0.0129 - learning_rate: 0.0010
Epoch 2/50
409/409 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step - loss: 3.9261e-04 - mae: 0.0132
Epoch 2: val_loss improved from 0.00020 to 0.00012, saving model to /kaggle/working/btc_model_outputs/model.keras
409/409 ━━━━━━━━━━━━━━━━━━━━ 57s 132ms/step - loss: 3.9351e-04 - mae: 0.0132 - val_loss: 1.2002e-04 - val_mae: 0.0105 - learning_rate: 0.0010
Epoch 3/50
409/409 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step - loss: 3.2035e-04 - mae: 0.0114
Epoch 3: val_loss did not improve from 0.00012
409/409 ━━━━━━━━━━━━━━━━━━━━ 56s 131ms/step - loss: 3.2118e-04 - mae: 0.0114 - val_loss: 1.7993e-04 - val_mae: 0

## 6. Optional Model Reload

This section can be used when the trained model needs to be loaded again after a Kaggle session restart.

Run this part only when the model has already been trained and saved in the working output directory.


In [ ]:
RUN_MODEL_RELOAD = False

if RUN_MODEL_RELOAD:
    WORK_DIR = Path("/kaggle/working")
    OUT_DIR = WORK_DIR / "btc_model_outputs"
    PROCESSED_DIR = WORK_DIR

    WINDOW_SIZE, HORIZON, BATCH_SIZE, CLOSE_IDX = 1440, 60, 32, 0

    model = tf.keras.models.load_model(OUT_DIR / "model.keras")

    try:
        scaler = joblib.load(PROCESSED_DIR / "scaler.pkl")
    except Exception:
        with open(PROCESSED_DIR / "scaler.pkl", "rb") as f:
            scaler = pickle.load(f)

    test_scaled = np.load(PROCESSED_DIR / "test_scaled.npy")

    test_ds = make_dataset(
        test_scaled,
        WINDOW_SIZE,
        HORIZON,
        BATCH_SIZE,
        shuffle=False,
        shift=60
    )

    print("Saved model and test dataset loaded successfully.")
else:
    print("Model reload skipped.")


## 7. Test Evaluation

This section evaluates the trained model on unseen test data.

The predictions are converted from scaled values back to USD so the final metrics are easier to interpret. The reported metrics include scaled MSE, scaled MAE, USD MAE, and USD RMSE.


In [ ]:
test_loss, test_mae_scaled = model.evaluate(test_ds, verbose=0)
print(f'  Test MSE (scaled) : {test_loss:.8f}')
print(f'  Test MAE (scaled) : {test_mae_scaled:.8f}')
print()

print('Collecting predictions...')
y_pred_scaled = []
y_true_scaled = []

for X_batch, y_batch in test_ds:
    preds = model.predict(X_batch, verbose=0)
    y_pred_scaled.extend(preds.flatten())
    y_true_scaled.extend(y_batch.numpy().flatten())

y_pred_scaled = np.array(y_pred_scaled)
y_true_scaled = np.array(y_true_scaled)

def inverse_transform_close(scaled_values, scaler, close_idx=0):
    n_features = scaler.n_features_in_
    dummy = np.zeros((len(scaled_values), n_features))
    dummy[:, close_idx] = scaled_values
    return scaler.inverse_transform(dummy)[:, close_idx]

y_pred_usd = inverse_transform_close(y_pred_scaled, scaler)
y_true_usd = inverse_transform_close(y_true_scaled, scaler)

errors   = y_pred_usd - y_true_usd
mae_usd  = float(np.mean(np.abs(errors)))
rmse_usd = float(np.sqrt(np.mean(errors ** 2)))
mse_usd  = float(np.mean(errors ** 2))

print()
print('=' * 40)
print('Test Set Results')
print('=' * 40)
print(f'  MAE  (USD) : ${mae_usd:,.2f}')
print(f'  RMSE (USD) : ${rmse_usd:,.2f}')
print(f'  MSE  (USD) : ${mse_usd:,.2f}')
print(f'  Bias (mean error) : ${errors.mean():+,.2f}')
print(f'  Actual price range: ${y_true_usd.min():,.0f} – ${y_true_usd.max():,.0f}')
print(f'  MAE as % of price : {mae_usd / y_true_usd.mean() * 100:.2f}%')
print('=' * 40)


## 8. Result Visualisation

This section visualises the model behaviour through training curves, predicted versus actual BTC prices, a zoomed prediction view, error distribution, and prediction scatter analysis.

These plots help evaluate convergence, prediction alignment, and error behaviour on the test set.


In [ ]:
has_history = "history" in globals()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

ax = axes[0, 0]
if has_history:
    ax.plot(history.history["loss"], label="Train loss")
    ax.plot(history.history["val_loss"], label="Validation loss")
    ax.set_yscale("log")
    ax.legend()
else:
    ax.text(0.5, 0.5, "Training history not available", ha="center", va="center")
ax.set_title("Training and Validation Loss (MSE)")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE")

ax = axes[0, 1]
ax.plot(y_true_usd, lw=1.0, label="Actual")
ax.plot(y_pred_usd, lw=1.0, label="Predicted", alpha=0.8)
ax.set_title("Actual vs Predicted — full test period")
ax.set_ylabel("USD")
ax.legend()

ax = axes[1, 0]
ax.plot(y_true_usd[:200], lw=1.5, label="Actual")
ax.plot(y_pred_usd[:200], lw=1.5, label="Predicted", alpha=0.8)
ax.set_title("Zoomed — first 200 predictions")
ax.set_ylabel("USD")
ax.set_xlabel("Hours")
ax.legend()

ax = axes[1, 1]
n_scatter = min(5000, len(y_true_usd))
ax.scatter(y_true_usd[:n_scatter], y_pred_usd[:n_scatter], s=3, alpha=0.2)
lims = [
    min(y_true_usd.min(), y_pred_usd.min()),
    max(y_true_usd.max(), y_pred_usd.max())
]
ax.plot(lims, lims, lw=1.2, linestyle="--", label="Perfect")
ax.set_title(f"Predicted vs Actual scatter, first {n_scatter}")
ax.set_xlabel("Actual (USD)")
ax.set_ylabel("Predicted (USD)")
ax.legend()

plt.suptitle(f"BTC Forecasting Model — MAE ${mae_usd:,.0f} | RMSE ${rmse_usd:,.0f}", fontsize=13)
plt.tight_layout()

results_path = OUT_DIR / "results.png"
plt.savefig(results_path, dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(errors, bins=100)
ax.axvline(0, linestyle="--", lw=1.5)
ax.set_title("Prediction Error Distribution (USD)")
ax.set_xlabel("Predicted − Actual (USD)")
ax.set_ylabel("Frequency")
ax.text(
    0.97, 0.95,
    f"Mean : ${errors.mean():+.2f}\nMAE  : ${mae_usd:,.2f}\nRMSE : ${rmse_usd:,.2f}",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=11,
    bbox=dict(boxstyle="round", facecolor="white", edgecolor="#cccccc")
)
plt.tight_layout()

error_dist_path = OUT_DIR / "error_dist.png"
plt.savefig(error_dist_path, dpi=150, bbox_inches="tight")
plt.show()

print("Saved:")
print(" ", results_path)
print(" ", error_dist_path)


## 9. Export Model Outputs

This section saves the prediction results and model-related outputs for later use.

The exported files can be used for reporting, dashboard visualisation, or deployment experiments.


In [ ]:
import json

print("Saving predictions...")

if df_features is not None:
    possible_time_cols = ["Timestamp", "timestamp", "Date", "date", "datetime", "Datetime"]
    time_col = next((c for c in possible_time_cols if c in df_features.columns), None)

    if time_col is not None:
        raw_times = pd.to_datetime(df_features[time_col], errors="coerce")
        raw_times = raw_times.dropna().reset_index(drop=True)

        if len(raw_times) >= len(y_true_usd):
            timestamps = raw_times.iloc[-len(y_true_usd):].dt.strftime("%Y-%m-%d %H:%M").tolist()
        else:
            timestamps = pd.date_range(
                start="2018-10-15",
                periods=len(y_true_usd),
                freq="60min"
            ).strftime("%Y-%m-%d %H:%M").tolist()
    else:
        timestamps = pd.date_range(
            start="2018-10-15",
            periods=len(y_true_usd),
            freq="60min"
        ).strftime("%Y-%m-%d %H:%M").tolist()
else:
    timestamps = pd.date_range(
        start="2018-10-15",
        periods=len(y_true_usd),
        freq="60min"
    ).strftime("%Y-%m-%d %H:%M").tolist()

predictions = {
    "metadata": {
        "window_size": WINDOW_SIZE,
        "horizon": HORIZON,
        "mae_usd": round(mae_usd, 2),
        "rmse_usd": round(rmse_usd, 2),
        "mse_scaled": round(float(test_loss), 8),
        "n_samples": len(y_true_usd),
        "architecture": "LSTM(128)->LSTM(64)->LSTM(32)->Dense(1)",
        "processed_dir": str(PROCESSED_DIR),
        "output_dir": str(OUT_DIR)
    },
    "data": [
        {
            "timestamp": timestamps[i],
            "actual": round(float(y_true_usd[i]), 2),
            "predicted": round(float(y_pred_usd[i]), 2),
            "error": round(float(errors[i]), 2)
        }
        for i in range(len(y_true_usd))
    ]
}

predictions_path = OUT_DIR / "predictions.json"
with open(predictions_path, "w") as f:
    json.dump(predictions, f, indent=2)

print(f"  predictions.json  → {predictions_path}")
print(f"  model.keras       → {OUT_DIR / 'model.keras'}")
print(f"  results.png       → {OUT_DIR / 'results.png'}")
print(f"  error_dist.png    → {OUT_DIR / 'error_dist.png'}")


## Final Summary

This notebook builds a complete BTC forecasting workflow:

- Loads preprocessed time-series data
- Creates efficient TensorFlow sequence datasets
- Trains a stacked LSTM forecasting model
- Evaluates predictions on unseen test data
- Converts predictions back to USD
- Generates visual diagnostics
- Saves the trained model and prediction outputs

The final model file and prediction artifacts are stored inside the Kaggle working output directory.
